# Full training on a Colab GPU

A laptop CPU needs ~5 s per training step, so the full runs (all of MASSIVE en/hi/ta + dialect data, 3 epochs) are done here.

1. Zip the project folder **without** `data/raw`, `data/processed` and `models` (they are regenerated / produced here) and upload it as `nlp_code.zip`.
2. Runtime -> Change runtime type -> GPU (T4 is enough).
3. Run all cells; download `trained_models.zip` at the end and unzip it into the project's `models/` folder.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose nlp_code.zip
!unzip -q -o nlp_code.zip -d /content/nlp_code_src && ls /content/nlp_code_src

In [ ]:
import os, glob
root = os.path.dirname(glob.glob('/content/nlp_code_src/**/requirements.txt', recursive=True)[0])
os.chdir(root); print(root)
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python scripts/download_data.py
!python scripts/prepare_data.py
!python scripts/train_dialect_id.py

In [ ]:
# Main model + ablation baseline (MuRIL), ~15-25 min each on a T4
!python scripts/train_nlu.py --encoder google/muril-base-cased --name muril_dialect --epochs 3
!python scripts/train_nlu.py --encoder google/muril-base-cased --name muril_std --no-dialects --epochs 3

In [ ]:
# Optional: encoder comparison (XLM-R vs MuRIL) and multi-task dialect head
!python scripts/train_nlu.py --encoder xlm-roberta-base --name xlmr_dialect --epochs 3
!python scripts/train_nlu.py --encoder google/muril-base-cased --name muril_multitask --dialect-head --epochs 3

In [ ]:
# Main path: muril_dialect without dialect->standard normalisation.
# --normalize adds ablation columns (helps muril_std only; not used in chat).
!python scripts/evaluate_nlu.py --models rules muril_std muril_dialect --normalize --tag full
!python scripts/evaluate_nlu.py --models muril_std muril_dialect --scope all --tag full
!python scripts/evaluate_context.py --models rules muril_std muril_dialect
# Optional experiment (not needed for chat):
# !python scripts/train_normalizer.py --epochs 8

In [ ]:
!zip -q -r trained_models.zip models reports
files.download('trained_models.zip')